# Module 5 - Defensive Federated Learning

Module 4 showed that malicious clients can damage naive FedAvg. Module 5 asks whether the server can reduce that damage without knowing which clients are malicious.

The server-side idea is simple: keep the same attack recipe, but stop blindly averaging every selected update. Your job in this lab is to compare plain FedAvg with defensive aggregation rules, then decide what each defense protects, what it costs, and where it can fail.


## 0. Module Framing

This notebook is the last step in the workshop sequence. Keep the earlier modules in view as you interpret each table and plot:

| Module | What you already built | What Module 5 does with it |
| --- | --- | --- |
| Module 1 | FedAvg loop | Uses FedAvg as the clean and attacked control condition. |
| Module 2 | Non-IID stress | Tests whether defenses still work when honest clients differ. |
| Module 3 | Optimizers | Keeps the focus on aggregation, not optimizer changes. |
| Module 4 | Malicious clients | Reuses the same poisoning path and metric vocabulary. |
| Module 5 | Defensive aggregation | Changes the server rule and measures whether the attack is reduced. |

Run the notebook top to bottom in the default minimal mode first. The longer sweeps are for workshop extensions or instructor demos.


### Concept Table

Use these terms consistently when you explain results. In particular, do not use a generic "ASR" unless the attacked model and success condition are named.

| Concept | Meaning in this notebook | What to inspect |
| --- | --- | --- |
| FedAvg | Plain average of selected client updates. It is the vulnerable control. | Clean and attacked FedAvg rows. |
| Clipping | Scales each client update to a maximum L2 norm before averaging. | Accuracy recovery and whether large malicious norms are limited. |
| Median | Takes the coordinate-wise median update across selected clients. | Robustness to extreme coordinate outliers. |
| Trimmed mean | Drops the largest and smallest coordinate values before averaging. | Sensitivity to `trim_fraction` and sampled-client count. |
| Krum | Selects one update whose nearest-neighbor distances are smallest. | Feasibility rule and whether one selected update is too restrictive. |
| Multi-Krum | Averages several Krum-selected updates. | Tradeoff between Krum robustness and using more honest signal. |
| Geometric median | Advanced robust center estimate, also called RFA in some papers. | Optional extension beyond the default defense list. |
| Surrogate poison success | `surrogate_poison_success_rate`: poisoned local examples that the attacker's MobileNetV2 surrogate predicts as the target label. | Per-round surrogate poison curves and final comparison bars. |
| Global target-label ASR | `global_target_label_asr`: held-out non-target test examples that the final MobileNetV3 global model predicts as the configured target label. | Global target-label ASR curves and final comparison bars. |
| Defense recovery | `defense_recovery`: defended final accuracy minus attacked FedAvg final accuracy. | Defense comparison table. |


## 1. Setup

This notebook can be run from the repository root or from inside `5_Defensive_FL/`. The setup cells load `config.yaml`, import the Module 4 attack-aware client path, create `artifacts/`, and validate the default defense assumptions before any training starts.


In [ ]:
from pathlib import Path
import json
import sys

import yaml

cwd = Path.cwd()
MODULE_DIR = cwd if cwd.name == "5_Defensive_FL" else cwd / "5_Defensive_FL"
REPO_ROOT = MODULE_DIR.parent
MODULE4_DIR = REPO_ROOT / "4_Adversarial_FL"

for path in (MODULE_DIR, MODULE4_DIR, REPO_ROOT):
    path_str = str(path.resolve())
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from defensive_servers import (
    make_attack_config,
    run_defensive_fl,
    sampled_client_count,
    validate_defense_config,
    validate_module5_config,
)
from metrics import build_comparison_rows, save_csv, save_json, update_norm_rows
from plots import (
    plot_accuracy_curves,
    plot_defense_comparison,
    plot_global_target_label_asr_curves,
    plot_sweep_metric,
    plot_surrogate_poison_success_curves,
    plot_update_norm_histogram,
)

ARTIFACT_DIR = MODULE_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = MODULE_DIR / "config.yaml"
with CONFIG_PATH.open("r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

issues = validate_module5_config(config, require_attack=True)
if issues:
    raise ValueError("Config validation failed:\n- " + "\n- ".join(issues))

sampled_clients = sampled_client_count(config["fed_config"])
print(
    "Module 5 config validated. "
    f"Each round samples {sampled_clients} clients from "
    f"{config['fed_config']['num_clients']} total clients "
    f"(fraction_clients={config['fed_config']['fraction_clients']})."
)

config


## 2. Threat Model Recap

The attacker controls some clients. The attacker can poison local training batches through Module 4's `MaliciousClient`. The attacker cannot edit the server directly. The server receives client updates and must aggregate them without knowing which selected clients are malicious.

This module does not add a stronger attacker. It holds the Module 4 attack recipe fixed so the aggregation rule is the main experimental variable.


## 3. Run Modes and Helpers

`RUN_MODE` controls how much work the notebook performs:

| Run mode | What runs | When to use it |
| --- | --- | --- |
| `minimal_demo` | Clean FedAvg, attacked FedAvg, defense comparison, plots, and validations. | Default student path. |
| `full_sweeps` | Everything in `minimal_demo` plus malicious-fraction, Krum, and non-IID sweeps. | Longer lab or instructor demo. |
| `load_existing` | Loads saved artifacts when present and skips training. | Review previous results without rerunning experiments. |

Longer sections also keep their own boolean flags so instructors can turn individual sweeps on without changing the rest of the notebook.


In [ ]:
global_config = config["global_config"]
data_config = config["data_config"]
fed_config = config["fed_config"]
model_config = config["model_config"]
optim_config = config.get("optim_config", {})
base_attack_config = config["attack"]

RUN_MODE = "minimal_demo"  # Options: "minimal_demo", "full_sweeps", "load_existing"
VALID_RUN_MODES = {"minimal_demo", "full_sweeps", "load_existing"}
if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(f"RUN_MODE must be one of {sorted(VALID_RUN_MODES)}.")

RUN_MINIMAL_DEMO = RUN_MODE in {"minimal_demo", "full_sweeps"}
RUN_FULL_SWEEPS = RUN_MODE == "full_sweeps"
LOAD_EXISTING_ONLY = RUN_MODE == "load_existing"

EXPECTED_ROUNDS = int(fed_config["num_rounds"])
REQUIRED_RESULT_METRICS = [
    "loss",
    "accuracy",
    "surrogate_poison_success_rate",
    "global_target_label_asr",
    "poisoned_examples",
    "candidate_examples",
    "sampled_malicious_clients",
    "defense_diagnostics",
    "round_runtime_sec",
]

COMPLETED_STATUS = "completed"
SKIPPED_STATUS = "skipped_infeasible"


def artifact_path(name):
    return ARTIFACT_DIR / name


def load_json_if_present(path):
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def completed_rows(rows):
    return [
        row for row in rows
        if row.get("status", COMPLETED_STATUS) == COMPLETED_STATUS
    ]


def validate_result(run_name, result, expected_rounds=EXPECTED_ROUNDS):
    missing = [metric for metric in REQUIRED_RESULT_METRICS if metric not in result]
    if missing:
        raise AssertionError(f"{run_name} is missing required metrics: {missing}")

    wrong_lengths = {}
    for metric in REQUIRED_RESULT_METRICS:
        values = result.get(metric)
        if not isinstance(values, list):
            wrong_lengths[metric] = "not a list"
        elif len(values) != expected_rounds:
            wrong_lengths[metric] = len(values)
    if wrong_lengths:
        raise AssertionError(
            f"{run_name} should log {expected_rounds} rounds for each metric; "
            f"got {wrong_lengths}"
        )

    return {
        "run": run_name,
        "rounds": expected_rounds,
        "final_accuracy": result["accuracy"][-1],
        "final_surrogate_poison_success_rate": result["surrogate_poison_success_rate"][-1],
        "final_global_target_label_asr": result["global_target_label_asr"][-1],
    }


def validate_result_collection(run_results, required_runs=None):
    if required_runs:
        missing_runs = [name for name in required_runs if name not in run_results]
        if missing_runs:
            raise AssertionError(f"Missing expected runs: {missing_runs}")
    return [
        validate_result(run_name, result)
        for run_name, result in run_results.items()
    ]


def validate_artifacts(paths):
    unique_paths = []
    seen = set()
    for path in paths:
        path = Path(path)
        if path not in seen:
            unique_paths.append(path)
            seen.add(path)
    missing = [path.name for path in unique_paths if not path.exists()]
    if missing:
        raise AssertionError(f"Expected artifacts were not saved: {missing}")
    return [path.name for path in unique_paths]


def defense_infeasibility_issues(defense_config, fed_config_override=None, context=None):
    active_fed_config = fed_config if fed_config_override is None else fed_config_override
    name = defense_config.get("name", "fedavg")
    return validate_defense_config(
        defense_config,
        active_fed_config,
        context=context or f"defense {name}",
    )


def make_skipped_row(run_name, defense_config, issues, fed_config_override=None, **metadata):
    active_fed_config = fed_config if fed_config_override is None else fed_config_override
    reason = "; ".join(issues)
    print(f"Skipping {run_name}: {reason}")
    row = {
        "run": run_name,
        "defense": defense_config.get("name", "fedavg"),
        "status": SKIPPED_STATUS,
        "skip_reason": reason,
        "sampled_clients": sampled_client_count(active_fed_config),
        "defense_config": dict(defense_config),
    }
    row.update(metadata)
    return row


def run_module5_experiment(
    run_name,
    attack_config,
    defense_config,
    data_config_override=None,
    fed_config_override=None,
):
    active_data_config = data_config if data_config_override is None else data_config_override
    active_fed_config = fed_config if fed_config_override is None else fed_config_override
    issues = defense_infeasibility_issues(
        defense_config,
        fed_config_override=active_fed_config,
        context=run_name,
    )
    if issues:
        raise ValueError("Infeasible defense config:\n- " + "\n- ".join(issues))

    server = run_defensive_fl(
        global_config=global_config,
        data_config=active_data_config,
        fed_config=active_fed_config,
        model_config=model_config,
        optim_config=optim_config,
        attack_config=attack_config,
        defense_config=defense_config,
    )
    result = server.results
    save_json(result, artifact_path(f"module5_{run_name}.json"))
    return result

print(f"RUN_MODE={RUN_MODE}; minimal_demo={RUN_MINIMAL_DEMO}; full_sweeps={RUN_FULL_SWEEPS}")


In [ ]:
required_sections = [
    "global_config",
    "data_config",
    "fed_config",
    "model_config",
    "attack",
    "experiments",
]
missing_sections = [section for section in required_sections if section not in config]
if missing_sections:
    raise AssertionError(f"Config is missing required sections: {missing_sections}")

if not config.get("experiments", {}).get("defenses"):
    raise AssertionError("config.yaml must list at least one defense in experiments.defenses.")

config_snapshot = {
    "config_path": str(CONFIG_PATH),
    "run_mode": RUN_MODE,
    "expected_rounds": EXPECTED_ROUNDS,
    "sampled_clients": sampled_clients,
    "experiment_defenses": config["experiments"]["defenses"],
    "attack": base_attack_config,
}
save_json(config_snapshot, artifact_path("module5_config_used.json"))

print(
    "Config loaded and recorded. "
    f"Expected rounds per run: {EXPECTED_ROUNDS}; "
    f"defenses listed: {[d['name'] for d in config['experiments']['defenses']]}"
)
config_snapshot


## 4. Baselines

Run a clean FedAvg baseline and an attacked FedAvg baseline before introducing defenses. By default, `RUN_MODE = "minimal_demo"` runs these baselines and the fixed-defense comparison. `RUN_MODE = "full_sweeps"` adds the longer sweeps, and `RUN_MODE = "load_existing"` skips training and reads saved artifacts when present.

The clean baseline answers: how well does the normal FL loop learn when no clients are malicious? The attacked baseline answers: how much damage does the Module 4 poisoning recipe cause when the server still uses plain FedAvg?


In [ ]:
RUN_BASELINES = RUN_MINIMAL_DEMO

run_results = {}

if RUN_BASELINES:
    clean_attack = make_attack_config(
        base_attack_config,
        malicious_fraction=0.0,
        poison_rate=0.0,
    )
    run_results["clean_fedavg"] = run_module5_experiment(
        "clean_fedavg",
        clean_attack,
        {"name": "fedavg"},
    )

    run_results["attacked_fedavg"] = run_module5_experiment(
        "attacked_fedavg",
        base_attack_config,
        {"name": "fedavg"},
    )
else:
    for run_name in ("clean_fedavg", "attacked_fedavg"):
        loaded = load_json_if_present(artifact_path(f"module5_{run_name}.json"))
        if loaded is not None:
            run_results[run_name] = loaded

run_results.keys()

In [ ]:
if RUN_BASELINES:
    required_baseline_runs = ["clean_fedavg", "attacked_fedavg"]
elif run_results:
    required_baseline_runs = None
else:
    required_baseline_runs = []

if required_baseline_runs or run_results:
    baseline_validation = validate_result_collection(
        run_results,
        required_runs=required_baseline_runs,
    )
    validate_artifacts(
        artifact_path(f"module5_{run_name}.json")
        for run_name in run_results
    )
    print(f"Validated {len(baseline_validation)} run result(s).")
else:
    baseline_validation = []
    print("No baseline runs were executed or loaded in this run mode.")

baseline_validation


### Interpret: Clean vs Attacked FedAvg

After the baseline cells finish, compare `clean_fedavg` with `attacked_fedavg` before looking at any defense:

1. Did attacked FedAvg lose clean-test accuracy relative to clean FedAvg?
2. Did `global_target_label_asr` increase after malicious clients began participating?
3. Is `surrogate_poison_success_rate` high even when the final global model does not show the same target-label behavior?
4. Which metric best explains why a server-side defense is needed?


## 5. Update Diagnostics

Before comparing defenses, inspect why FedAvg is vulnerable. Update norm, cosine-to-mean, and distance-to-median diagnostics are stored for every round of the attacked FedAvg run.

These diagnostics do not identify attackers perfectly. They give evidence about whether malicious updates look unusually large, unusually directional, or far from the coordinate-wise center of the selected client updates.


In [ ]:
if "attacked_fedavg" in run_results:
    norm_rows = update_norm_rows(run_results["attacked_fedavg"])
    save_json(norm_rows, artifact_path("module5_update_diagnostics.json"))
    plot_update_norm_histogram(
        norm_rows,
        artifact_path("module5_update_norms.png"),
        round_number=base_attack_config["start_round"],
    )
else:
    print("Run or load attacked_fedavg before plotting update diagnostics.")

### Interpret: Update Norm Diagnostics

Use `module5_update_diagnostics.json` and `module5_update_norms.png` to reason about the attack geometry:

1. In the attack-start round, do malicious updates have larger L2 norms than honest updates?
2. Are malicious updates separated by cosine-to-mean or distance-to-median, even if norms overlap?
3. Which defense is each diagnostic motivating: clipping, median or trimmed mean, Krum, or Multi-Krum?
4. What would happen if honest non-IID updates looked just as unusual as malicious updates?


## 6. Defense Comparison

Run each defense under the same Module 4 attack recipe. The comparison records `surrogate_poison_success_rate` separately from final `global_target_label_asr` so surrogate behavior is not mistaken for global-model attack success.

Treat FedAvg as the attacked control row. A useful defense should recover accuracy, reduce global target-label behavior, or both, without relying on information the server would not have in a real deployment.


In [ ]:
RUN_DEFENSE_COMPARISON = RUN_MINIMAL_DEMO

skipped_defense_rows = []
if RUN_DEFENSE_COMPARISON:
    for defense_config in config["experiments"]["defenses"]:
        name = defense_config["name"]
        if name == "fedavg" and "attacked_fedavg" in run_results:
            continue
        issues = defense_infeasibility_issues(
            defense_config,
            context=f"defense comparison {name}",
        )
        if issues:
            skipped_defense_rows.append(
                make_skipped_row(name, defense_config, issues)
            )
            continue
        run_results[name] = run_module5_experiment(
            name,
            base_attack_config,
            defense_config,
        )
else:
    for defense_config in config["experiments"]["defenses"]:
        name = "attacked_fedavg" if defense_config["name"] == "fedavg" else defense_config["name"]
        loaded = load_json_if_present(artifact_path(f"module5_{name}.json"))
        if loaded is not None:
            run_results[name] = loaded

comparison_rows = build_comparison_rows(run_results)
for row in comparison_rows:
    row.setdefault("status", COMPLETED_STATUS)
comparison_rows.extend(skipped_defense_rows)

expected_defense_runs = [
    "attacked_fedavg"
    if defense["name"] == "fedavg" and "attacked_fedavg" in run_results
    else defense["name"]
    for defense in config["experiments"]["defenses"]
]
seen_defense_runs = {row.get("run") for row in comparison_rows}
if RUN_DEFENSE_COMPARISON:
    missing_defense_runs = [
        run_name for run_name in expected_defense_runs
        if run_name not in seen_defense_runs
    ]
    if missing_defense_runs:
        raise AssertionError(
            "Defense comparison did not record every configured defense: "
            f"{missing_defense_runs}"
        )

if comparison_rows:
    save_json(comparison_rows, artifact_path("module5_defense_comparison.json"))
    save_csv(comparison_rows, artifact_path("module5_defense_comparison.csv"))
else:
    print("No defense comparison rows yet. Run baselines or load prior artifacts first.")
comparison_rows


In [ ]:
plot_rows = completed_rows(comparison_rows)
if plot_rows:
    plot_accuracy_curves(
        run_results,
        artifact_path("module5_accuracy_curves.png"),
        attack_start_round=base_attack_config["start_round"],
    )
    plot_surrogate_poison_success_curves(
        run_results,
        artifact_path("module5_surrogate_poison_success_curves.png"),
        attack_start_round=base_attack_config["start_round"],
    )
    plot_global_target_label_asr_curves(
        run_results,
        artifact_path("module5_global_target_label_asr_curves.png"),
        attack_start_round=base_attack_config["start_round"],
    )
    plot_defense_comparison(
        plot_rows,
        metric="final_accuracy",
        path=artifact_path("module5_accuracy_vs_defense.png"),
        ylabel="Final accuracy (%)",
        title="Final accuracy by defense",
    )
    plot_defense_comparison(
        plot_rows,
        metric="final_surrogate_poison_success_rate",
        path=artifact_path("module5_surrogate_poison_success_vs_defense.png"),
        ylabel="Final surrogate poison success rate (%)",
        title="Final surrogate poison success rate by defense",
    )
    plot_defense_comparison(
        plot_rows,
        metric="final_global_target_label_asr",
        path=artifact_path("module5_global_target_label_asr_vs_defense.png"),
        ylabel="Final global target-label ASR (%)",
        title="Final global target-label ASR by defense",
    )
else:
    print("No completed comparison rows available yet.")


In [ ]:
completed_comparison_rows = completed_rows(comparison_rows)
if completed_comparison_rows:
    comparison_validation = validate_result_collection(run_results)
    comparison_artifacts = [
        artifact_path("module5_defense_comparison.json"),
        artifact_path("module5_defense_comparison.csv"),
        artifact_path("module5_accuracy_curves.png"),
        artifact_path("module5_surrogate_poison_success_curves.png"),
        artifact_path("module5_global_target_label_asr_curves.png"),
        artifact_path("module5_accuracy_vs_defense.png"),
        artifact_path("module5_surrogate_poison_success_vs_defense.png"),
        artifact_path("module5_global_target_label_asr_vs_defense.png"),
    ]
    validate_artifacts(comparison_artifacts)
    print(
        f"Validated {len(completed_comparison_rows)} completed comparison row(s) "
        f"and {len(comparison_artifacts)} comparison artifact(s)."
    )
else:
    comparison_validation = []
    print("No completed defense comparison rows available to validate.")

comparison_validation


### Interpret: Defense Comparison

Read the comparison table and plots together:

1. Which defense has the largest positive `defense_recovery`?
2. Which defense most reduces `final_global_target_label_asr` relative to attacked FedAvg?
3. Did any defense reduce target-label behavior while also hurting final accuracy?
4. Do `final_surrogate_poison_success_rate` and `final_global_target_label_asr` tell the same story or different stories?
5. If Krum or Multi-Krum is skipped, what sampled-client feasibility rule caused the skip?


## 7. Malicious Fraction Sweep

This sweep shows where each defense starts to break as the malicious-client fraction increases. It is guarded by `RUN_MALICIOUS_FRACTION_SWEEP`, which is `False` in the default minimal demo and `True` only when `RUN_MODE = "full_sweeps"` unless you override it manually.


In [ ]:
RUN_MALICIOUS_FRACTION_SWEEP = RUN_FULL_SWEEPS

sweep_rows = []
if RUN_MALICIOUS_FRACTION_SWEEP:
    for malicious_fraction in config["experiments"]["malicious_fraction_sweep"]:
        attack_config = make_attack_config(base_attack_config, malicious_fraction=malicious_fraction)
        for defense_config in config["experiments"]["defenses"]:
            run_name = f"{defense_config['name']}_mf_{malicious_fraction}"
            issues = defense_infeasibility_issues(defense_config, context=run_name)
            if issues:
                sweep_rows.append(
                    make_skipped_row(
                        run_name,
                        defense_config,
                        issues,
                        malicious_fraction=malicious_fraction,
                    )
                )
                continue

            result = run_module5_experiment(run_name, attack_config, defense_config)
            row = build_comparison_rows({run_name: result})[0]
            row["defense"] = defense_config["name"]
            row["malicious_fraction"] = malicious_fraction
            row["status"] = COMPLETED_STATUS
            sweep_rows.append(row)

    save_json(sweep_rows, artifact_path("module5_malicious_fraction_sweep.json"))
    completed_sweep_rows = completed_rows(sweep_rows)
    if completed_sweep_rows:
        plot_sweep_metric(
            completed_sweep_rows,
            x_key="malicious_fraction",
            y_key="final_accuracy",
            group_key="defense",
            path=artifact_path("module5_malicious_fraction_accuracy.png"),
            ylabel="Final accuracy (%)",
        )
        plot_sweep_metric(
            completed_sweep_rows,
            x_key="malicious_fraction",
            y_key="final_global_target_label_asr",
            group_key="defense",
            path=artifact_path("module5_malicious_fraction_global_target_label_asr.png"),
            ylabel="Final global target-label ASR (%)",
        )
    else:
        print("No feasible malicious-fraction sweep rows to plot.")
else:
    sweep_rows = load_json_if_present(artifact_path("module5_malicious_fraction_sweep.json")) or []

sweep_rows[:3]


### Interpret: Malicious-Fraction Sweep

When this sweep is enabled, focus on the shape of each defense curve rather than a single number:

1. Does a defense fail gradually as malicious fraction rises, or does it collapse after a threshold?
2. At which malicious fraction does attacked FedAvg become unacceptable for your accuracy target?
3. Does lower `global_target_label_asr` come with a loss in clean-test accuracy?
4. Which defense would you choose if the malicious fraction were uncertain?


## 8. Krum Hyperparameter Sweep

Krum and Multi-Krum need enough participating clients for the configured Byzantine tolerance. This sweep intentionally keeps infeasible settings visible as skipped rows instead of failing midway through notebook execution.


In [ ]:
RUN_KRUM_HYPERPARAMETER_SWEEP = RUN_FULL_SWEEPS

krum_sweep_rows = []
if RUN_KRUM_HYPERPARAMETER_SWEEP:
    for byzantine_f in config["experiments"]["krum_byzantine_f_sweep"]:
        defense_config = {"name": "krum", "byzantine_f": byzantine_f}
        run_name = f"krum_f_{byzantine_f}"
        issues = defense_infeasibility_issues(defense_config, context=run_name)
        if issues:
            krum_sweep_rows.append(
                make_skipped_row(
                    run_name,
                    defense_config,
                    issues,
                    byzantine_f=byzantine_f,
                )
            )
            continue

        result = run_module5_experiment(run_name, base_attack_config, defense_config)
        row = build_comparison_rows({run_name: result})[0]
        row["defense"] = "krum"
        row["byzantine_f"] = byzantine_f
        row["status"] = COMPLETED_STATUS
        krum_sweep_rows.append(row)

    save_json(krum_sweep_rows, artifact_path("module5_krum_byzantine_f_sweep.json"))
    completed_krum_rows = completed_rows(krum_sweep_rows)
    if completed_krum_rows:
        plot_sweep_metric(
            completed_krum_rows,
            x_key="byzantine_f",
            y_key="final_accuracy",
            group_key="defense",
            path=artifact_path("module5_krum_byzantine_f_accuracy.png"),
            ylabel="Final accuracy (%)",
        )
        plot_sweep_metric(
            completed_krum_rows,
            x_key="byzantine_f",
            y_key="final_global_target_label_asr",
            group_key="defense",
            path=artifact_path("module5_krum_byzantine_f_global_target_label_asr.png"),
            ylabel="Final global target-label ASR (%)",
        )
    else:
        print("No feasible Krum hyperparameter sweep rows to plot.")
else:
    krum_sweep_rows = load_json_if_present(artifact_path("module5_krum_byzantine_f_sweep.json")) or []

krum_sweep_rows


### Interpret: Krum Feasibility

Use skipped rows as part of the result. Krum is not just an aggregation formula; it has a sampled-client assumption.

1. Which `byzantine_f` values are feasible under the current `num_clients` and `fraction_clients`?
2. Does increasing `byzantine_f` improve robustness, or does selecting too few updates hurt learning?
3. What configuration change would make a skipped Krum setting feasible?


## 9. Non-IID Stress Test

Non-IID data can make honest client updates look like outliers. This stress test asks whether each defense still helps when client data distributions are less similar. It is guarded by `RUN_NON_IID_STRESS`, so the default minimal demo does not launch the long stress test.


In [ ]:
RUN_NON_IID_STRESS = RUN_FULL_SWEEPS

non_iid_rows = []
if RUN_NON_IID_STRESS:
    selected_defenses = [
        {"name": "fedavg"},
        {"name": "clipping", "clip_norm": 5.0},
        {"name": "trimmed_mean", "trim_fraction": 0.1},
        {"name": "krum", "byzantine_f": 2},
    ]
    for non_iid_per in config["experiments"]["non_iid_sweep"]:
        data_variant = dict(data_config)
        data_variant["non_iid_per"] = non_iid_per
        for defense_config in selected_defenses:
            run_name = f"{defense_config['name']}_noniid_{non_iid_per}"
            issues = defense_infeasibility_issues(defense_config, context=run_name)
            if issues:
                non_iid_rows.append(
                    make_skipped_row(
                        run_name,
                        defense_config,
                        issues,
                        non_iid_per=non_iid_per,
                    )
                )
                continue

            result = run_module5_experiment(
                run_name,
                base_attack_config,
                defense_config,
                data_config_override=data_variant,
            )
            row = build_comparison_rows({run_name: result})[0]
            row["defense"] = defense_config["name"]
            row["non_iid_per"] = non_iid_per
            row["status"] = COMPLETED_STATUS
            non_iid_rows.append(row)

    save_json(non_iid_rows, artifact_path("module5_non_iid_defense_stress.json"))
    completed_non_iid_rows = completed_rows(non_iid_rows)
    if completed_non_iid_rows:
        plot_sweep_metric(
            completed_non_iid_rows,
            x_key="non_iid_per",
            y_key="final_accuracy",
            group_key="defense",
            path=artifact_path("module5_non_iid_accuracy.png"),
            ylabel="Final accuracy (%)",
        )
        plot_sweep_metric(
            completed_non_iid_rows,
            x_key="non_iid_per",
            y_key="final_global_target_label_asr",
            group_key="defense",
            path=artifact_path("module5_non_iid_global_target_label_asr.png"),
            ylabel="Final global target-label ASR (%)",
        )
    else:
        print("No feasible non-IID stress rows to plot.")
else:
    non_iid_rows = load_json_if_present(artifact_path("module5_non_iid_defense_stress.json")) or []

non_iid_rows[:3]


### Interpret: Non-IID Stress Test

When this stress test is enabled, connect the result back to Module 2:

1. Which defenses mistake honest heterogeneity for malicious behavior?
2. Does FedAvg look better under non-IID data than it did under the fixed attack comparison, or worse?
3. Do robust defenses preserve enough honest signal when clients hold different label distributions?
4. What failure mode would you warn a practitioner about before deploying the best-looking defense?


## 10. Final Synthesis

Use the saved tables and plots to answer the defensive FL question in one short paragraph:

> Which aggregation rule would you choose under this threat model, and what evidence supports that choice?

A complete answer should reference clean accuracy, attacked accuracy, `surrogate_poison_success_rate`, `global_target_label_asr`, and `defense_recovery`. It should also name at least one failure mode, such as infeasible Krum settings, aggressive clipping, or non-IID honest clients that resemble outliers.

Workshop synthesis:

| Module | Synthesis point |
| --- | --- |
| Module 1 | FedAvg gives the basic client-server training loop. |
| Module 2 | Non-IID data makes even honest FL harder to interpret. |
| Module 3 | Optimizers can improve convergence but do not by themselves solve malicious updates. |
| Module 4 | Malicious clients can poison FedAvg, and surrogate poison success is not the same as final global-model target-label behavior. |
| Module 5 | Defensive aggregation can reduce attack impact, but every defense depends on assumptions about update geometry, client sampling, and honest data heterogeneity. |

Before leaving the module, inspect the final artifact validation below. Those files are the evidence for your written interpretation.


In [ ]:
expected_artifacts = [artifact_path("module5_config_used.json")]

for run_name in run_results:
    expected_artifacts.append(artifact_path(f"module5_{run_name}.json"))

if "attacked_fedavg" in run_results:
    expected_artifacts.extend([
        artifact_path("module5_update_diagnostics.json"),
        artifact_path("module5_update_norms.png"),
    ])

if comparison_rows:
    expected_artifacts.extend([
        artifact_path("module5_defense_comparison.json"),
        artifact_path("module5_defense_comparison.csv"),
    ])
    if completed_rows(comparison_rows):
        expected_artifacts.extend([
            artifact_path("module5_accuracy_curves.png"),
            artifact_path("module5_surrogate_poison_success_curves.png"),
            artifact_path("module5_global_target_label_asr_curves.png"),
            artifact_path("module5_accuracy_vs_defense.png"),
            artifact_path("module5_surrogate_poison_success_vs_defense.png"),
            artifact_path("module5_global_target_label_asr_vs_defense.png"),
        ])

if sweep_rows:
    expected_artifacts.append(artifact_path("module5_malicious_fraction_sweep.json"))
    if completed_rows(sweep_rows):
        expected_artifacts.extend([
            artifact_path("module5_malicious_fraction_accuracy.png"),
            artifact_path("module5_malicious_fraction_global_target_label_asr.png"),
        ])

if krum_sweep_rows:
    expected_artifacts.append(artifact_path("module5_krum_byzantine_f_sweep.json"))
    if completed_rows(krum_sweep_rows):
        expected_artifacts.extend([
            artifact_path("module5_krum_byzantine_f_accuracy.png"),
            artifact_path("module5_krum_byzantine_f_global_target_label_asr.png"),
        ])

if non_iid_rows:
    expected_artifacts.append(artifact_path("module5_non_iid_defense_stress.json"))
    if completed_rows(non_iid_rows):
        expected_artifacts.extend([
            artifact_path("module5_non_iid_accuracy.png"),
            artifact_path("module5_non_iid_global_target_label_asr.png"),
        ])

artifact_validation = validate_artifacts(expected_artifacts)
print(f"Validated {len(artifact_validation)} expected artifact(s) for this run mode.")
artifact_validation
